# Multi Schema用法
### 1. 状态类型
LangGraph 支持在一个图中使用多个状态Schema，用于区分图的外部输入、外部输出、内部共享状态以及节点间的临时状态。
常见状态类型包括：
- 全局状态/内部状态：图内部主要使用的状态，创建StateGraph时传递给state_schema参数。通常包含图运行过程中需要读写的大部分字段。
- 输入状态：图对外接收输入时使用的状态，创建StateGraph是传递给input_schema参数。它用于约束调用图时允许传入哪些字段。（可以看做全局状态的一部分）
- 输出状态：图最终对外返回结果时使用的状态，创建StateGraph时传递给output_schema参数。用于约束图运行结束后只返回哪些字段。
- 私有状态：图内部节点之间传递的临时状态，通常不作为图的输入，也不作为图的最终输出。可以通过节点函数的入参类型注解声明，并在节点返回值中写入。
Tips:
    输入状态和输出状态主要面向图的边界，即“图如何接收外部输入”和“图如何返回外部输出”；而全局状态和私有状态主要面向图内部节点之间的数据传递。


### 2. 设计规范
- 2.1 输入状态描述图对外需要接收的数据，输出状态描述图最终需要返回的数据。通常情况，它们都应该是全局状态的一部分。
- 2.2 私有状态和全局状态应尽量避免字段重名。
- 2.3 节点函数应明确声明入参状态类型和返回状态类型。
- 2.4 节点函数中不应该访问入参状态类型中不存在的字段。
- 2.5 节点函数返回的字典应尽量和返回类型注解保持一致。

### 3. 源码层面的约束
- 3.1 状态的记录
    - LangGraph 的状态不是简单保存在一个普通字典中，而是会被拆分成多个可读写的状态字段。每个状态字段在底层通常对应一个Channel。
    - 3.1.1 StateGraph 记录状态字段的核心方法是 _add_schema()
    - 3.1.2 创建StateGraph时，会记录state_schema、input_schema和output_schema中的字段。
    例：
    ```python
    builder = StateGraph(
        OverAllState,
        input_schema=InputSchema,
        output_schema=OutputSchema,
    )
    ```
    LangGraph 会解析这些Schema，并将其中涉及的字段加入图的状态管理体系。
    - 3.1.3 调用 add_node() 添加节点时，也可能记录节点入参声明的状态Schema
    例：
    '''python
    class PrivateState(TypedDict):
        greeting: str

    def node_3(state: PrivateState) -> OutputState:
        return {
            "graph_output": state["greeting"]
        }
    '''
    当 node_3 被添加到图中时，PrivateState 中的greeting 字段会被记录到图中，从而成为图内部可以传递的状态字段。
    - 3.1.4 总结
        - 全局状态、输入状态、输出状态通常在创建StateGraph时被记录。
        - 私有状态通常在调用 add_node() 添加节点时，根据节点入参类型注解被记录。
        - 被记录后的状态字段，底层会成为图运行时可以读写的状态字段。
- 3.2 状态的访问
    - 3.2.1 调用图时，输入会按照 input_schema 进行约束
    - 3.2.2 节点接收到的状态会按照节点入参类型进行裁剪
    - 3.2.3 节点返回的是状态更新，而不是完整状态。
    - 3.2.4 几点返回值的应用主要由字段名称和图中已记录的状态字段决定。
    - 3.2.5 最终输出会按照 output_schema 进行裁剪。



In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# 1. 输入状态
class InputState(TypedDict):
    username: str

# 2. 输出状态
class OutputState(TypedDict):
    graph_output: str

# 3. 全局状态
class OverAllState(TypedDict):
    username: str
    graph_output: str
    nickname: str

# 4. 私有状态
class PrivateState(TypedDict):
    greeting: str


# 5. 定义第一个节点 node_1 对接START => InputState
def node_1(state: InputState) -> OverAllState:
    '''
    向全局状态添加username
    '''
    return {
        "nickname": "Dear " + state["username"]
    }

# 6. 定义第二个节点 node_2 对接OverAllState => PrivateState
def node_2(state: OverAllState) -> PrivateState:
    '''
    向私有状态添加greeting
    '''
    return {
        "greeting": "Hello, " + state["nickname"]
    }

# 7. 定义第三个节点 node_3 对接PrivateState => OutputState
def node_3(state: PrivateState) -> OutputState:
    '''
    向输出状态添加g_output
    '''
    return {
        "graph_output": state["greeting"] + " 很高兴认识你！"
    }

# 8. 构建状态图
builder = StateGraph(state_schema=OverAllState, input_schema=InputState, output_schema=OutputState)

# 9. 添加节点
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)

# 10. 添加边
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", "node_3")
builder.add_edge("node_3", END)

# 11. 编译状态图
graph = builder.compile()
result = graph.invoke({"username": "张三"})
print('=' * 30,'-> result <-', '=' * 30)
print(result)

============================== -> result <- ==============================
{'graph_output': 'Hello, Dear 张三 很高兴认识你！'}
